## CIS 9655 — Two Truths and a Lie
**Student:** Xixi Lin  
**Dataset:** World Bank GDP per Capita (1960–2024)  
**Questions:** 
1. How has the GDP per capita gap between income groups widened over time (1960–2024)?  
2. Which countries had the highest GDP per capita growth (1960–2024)?

In [91]:
#step 1 install plotly
%pip install plotly nbformat


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3.14 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [92]:
#step2 import 
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import plotly.io as pio

from plotly.subplots import make_subplots

import sys
!{sys.executable} -m pip install requests

pio.renderers.default = "plotly_mimetype"

px.defaults.template = "plotly_white"
#px.defaults.width = 900
#px.defaults.height = 550



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3.14 install --upgrade pip


In [93]:
import pandas as pd
import requests
from io import StringIO
# ── import the files─────────────────────────────────────────────────────
GDP_file = pd.read_csv("./API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46.csv", skiprows=4)
Meta_file = pd.read_csv("./Metadata_Country_API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46.csv")[["Country Code", "Region", "IncomeGroup"]]
data_GDP = pd.merge(GDP_file, Meta_file, on="Country Code", how="inner")

# ── Clean ─────────────────────────────────────────────────────────────────────
# Drop columns we don't need
data_GDP = data_GDP.drop(columns=["Indicator Name", "Indicator Code", "Unnamed: 65"], errors="ignore")

# Keep only actual countries (rows with a Region value — drops aggregates like "Africa Eastern and Southern")
data_GDP = data_GDP[data_GDP["Region"].notna() & (data_GDP["Region"] != "")]

# Melt from wide (years as columns) → long format (one row per country-year)
year_cols = [str(y) for y in range(1960, 2025)]
df1 = data_GDP.melt(
    id_vars=["Country Name", "Country Code", "Region", "IncomeGroup"],
    value_vars=year_cols,
    var_name="Year",
    value_name="GDP_per_capita"
)

# Clean up types
df1["Year"] = df1["Year"].astype(int)
df1["GDP_per_capita"] = pd.to_numeric(df1["GDP_per_capita"], errors="coerce")

# Drop rows with no GDP value
df1 = df1.dropna(subset=["GDP_per_capita"])

print(df1.shape)
print(df1.head())
print(df1["Region"].unique())
print(df1["IncomeGroup"].unique())

(11569, 6)
   Country Name Country Code                     Region          IncomeGroup  \
6     Argentina          ARG  Latin America & Caribbean  Upper middle income   
10    Australia          AUS        East Asia & Pacific          High income   
11      Austria          AUT      Europe & Central Asia          High income   
13      Burundi          BDI         Sub-Saharan Africa           Low income   
14      Belgium          BEL      Europe & Central Asia          High income   

    Year  GDP_per_capita  
6   1960      778.251707  
10  1960     1813.431099  
11  1960      939.914815  
13  1960       70.905100  
14  1960     1290.286072  
<StringArray>
[ 'Latin America & Caribbean',        'East Asia & Pacific',
      'Europe & Central Asia',         'Sub-Saharan Africa',
                 'South Asia',              'North America',
 'Middle East & North Africa']
Length: 7, dtype: str
<StringArray>
['Upper middle income', 'High income', 'Low income', 'Lower middle income',
 nan]


<Analytical Question Headline. Max one line.>
<Question details or accompanying explanatory text. Max two lines. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Ut rhoncus lorem dolor, sit amet
ullamcorper. Ut ultricies, orci in dignissim lobortis, augue purus rhoncus purus, in dictum magna lorem ac.>

Analytical Question Headline. Max one line.

In [94]:
df1
#prefer this way to display data, for student only

,Country Name,Country Code,Region,IncomeGroup,Year,GDP_per_capita
6,Argentina,ARG,Latin America & Caribbean,Upper middle income,1960,778.251707
10,Australia,AUS,East Asia & Pacific,High income,1960,1813.431099
11,Austria,AUT,Europe & Central Asia,High income,1960,939.914815
13,Burundi,BDI,Sub-Saharan Africa,Low income,1960,70.905100
14,Belgium,BEL,Europe & Central Asia,High income,1960,1290.286072
...,...,...,...,...,...,...
14099,Samoa,WSM,East Asia & Pacific,Upper middle income,2024,5392.877619
14100,Kosovo,XKX,Europe & Central Asia,Upper middle income,2024,7023.065985
14102,South Africa,ZAF,Sub-Saharan Africa,Upper middle income,2024,6267.186814
14103,Zambia,ZMB,Sub-Saharan Africa,Lower middle income,2024,1187.109434


<h3>Question 1: How has the GDP per capita gap between income groups widened over time (1960–2024)?</h3>

<h5>The analytical task is to compare trends over time across different income groups to identify divergence and specific inflection points. A line chart is a strong fit because it excels at showing continuity and the rate of change over a temporal axis (x-axis). It allows for immediate visual comparison of multiple series (income groups) to see if they are converging or diverging. An alternative would be a Grouped Bar Chart. However, it would be weaker because with over 60 years of data, the chart would become extremely cluttered, making it difficult to perceive the overall long-term trend and the specific 'breakout' moment in 1985.</h5>


In [95]:


# ── Average GDP per capita by income group and year ───────────────────────────
df_income = df1.groupby(["IncomeGroup", "Year"])["GDP_per_capita"].mean().reset_index()
df_income = df_income[df_income["Year"] >= 1960].copy()

income_cols = ["High income", "Upper middle income", "Lower middle income", "Low income"]
df_income["IncomeGroup"] = pd.Categorical(
    df_income["IncomeGroup"],
    categories=income_cols,
    ordered=True
)

df_pivot = df_income.pivot(index="Year", columns="IncomeGroup", values="GDP_per_capita").reset_index()

# ── Rolling standard deviation method to find breakout year ───────────────────
df_pivot["std_gap"] = df_pivot[income_cols].std(axis=1)
df_pivot["std_gap_smooth"] = df_pivot["std_gap"].rolling(window=5, center=True).mean()
df_pivot["std_acceleration"] = df_pivot["std_gap_smooth"].diff().diff()

# Find breakout year from pre-2000 data
df_pre2000 = df_pivot[df_pivot["Year"] <= 2000].copy()
key_year = int(df_pre2000.loc[df_pre2000["std_acceleration"].idxmax(), "Year"])
key_gap = df_pivot.loc[df_pivot["Year"] == key_year, "std_gap"].values[0]

print(f"Breakout Year: {key_year}")
print(f"Std gap at breakout: ${key_gap:,.0f}")

# ── Values for start/end gap annotations ──────────────────────────────────────
val_high_1960 = df_pivot.loc[df_pivot["Year"] == 1960, "High income"].values[0]
val_low_1960 = df_pivot.loc[df_pivot["Year"] == 1960, "Low income"].values[0]
val_high_2024 = df_pivot.loc[df_pivot["Year"] == 2024, "High income"].values[0]
val_low_2024 = df_pivot.loc[df_pivot["Year"] == 2024, "Low income"].values[0]

gap_1960 = val_high_1960 - val_low_1960
gap_2024 = val_high_2024 - val_low_2024

high_growth = val_high_2024 / val_high_1960
low_growth = val_low_2024 / val_low_1960

# ── Plot ───────────────────────────────────────────────────────────────────────
fig = px.line(
    df_income.sort_values(["IncomeGroup", "Year"]),
    x="Year",
    y="GDP_per_capita",
    color="IncomeGroup",
    markers=False,
    labels={
        "GDP_per_capita": "Avg GDP per Capita (US$)",
        "Year": "Year",
        "IncomeGroup": "Income Group"
    },
    color_discrete_map={
        "High income": "#1a56db",
        "Upper middle income": "#45a049",
        "Lower middle income": "#f5a623",
        "Low income": "#d9534f"
    },
    category_orders={"IncomeGroup": income_cols}
)

# ── Breakout year line ─────────────────────────────────────────────────────────
fig.add_vline(
    x=key_year,
    line_dash="dash",
    line_color="gray",
    line_width=1.5
)

fig.add_annotation(
    x=key_year,
    y=df_pivot["High income"].max() * 0.88,
    text=f"Breakout: {key_year}",
    showarrow=True,
    arrowhead=2,
    ax=35,
    ay=0,
    font=dict(size=12, color="darkred")
)

# ── Gap annotations ────────────────────────────────────────────────────────────
fig.add_annotation(
    x=1960,
    y=(val_high_1960 + val_low_1960) / 2,
    text=f"Gap: ${gap_1960:,.0f}",
    showarrow=True,
    arrowhead=2,
    ax=-70,
    ay=0,
    font=dict(size=11, color="#1a56db"),
    bgcolor="white",
    bordercolor="#1a56db",
    borderwidth=1
)

fig.add_annotation(
    x=2024,
    y=(val_high_2024 + val_low_2024) / 2,
    text=f"Gap: ${gap_2024:,.0f}",
    showarrow=True,
    arrowhead=2,
    ax=70,
    ay=0,
    font=dict(size=11, color="#1a56db"),
    bgcolor="white",
    bordercolor="#1a56db",
    borderwidth=1
)

# ── Subtitle above chart ───────────────────────────────────────────────────────
fig.add_annotation(
    x=0.5,
    y=1.14,
    xref="paper",
    yref="paper",
    showarrow=False,
    align="left",
    text=(
        f"From 1960 to 2024, the GDP per capita gap widened from <b>${gap_1960:,.0f}</b> "
        f"to <b>${gap_2024:,.0f}</b>.<br>"
        f"High income countries grew {high_growth:.1f}× — from ${val_high_1960:,.0f} to ${val_high_2024:,.0f} — "
        f"while low income countries grew only {low_growth:.1f}×.<br>"
        f"The divide accelerated sharply after <b>{key_year}</b>,"
        f" driven by globalization and the digital revolution, leaving the world's poorest nations far behind.</b>"
    ),
    font=dict(size=13, color="#2f4f6f"),
)

# ── Bottom takeaway ────────────────────────────────────────────────────────────
fig.add_annotation(
    x=0.5,
    y=-0.20,
    xref="paper",
    yref="paper",
    text=(
        f"<b>Takeaway:</b> High income countries grew ~47x since 1960. Low income countries grew only ~7x. The gap has accelerated since 1985 <br> "
    ),
    showarrow=False,
    font=dict(size=11, color="gray"),
    align="center"
)

# ── Trace formatting ───────────────────────────────────────────────────────────
fig.update_traces(
    mode="lines",
    hovertemplate="<b>%{fullData.name}</b><br>Year: %{x}<br>GDP per Capita: $%{y:,.0f}<extra></extra>"
)

# ── Layout ─────────────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text="How has the GDP per capita gap between income groups changed over time (1960–2024)?",
        x=0.01,
        xanchor="left",
        y=0.97,
        yanchor="top",              
        font=dict(size=20)
    ),
    hovermode="closest",
    xaxis_title="Year",
    yaxis_title="Avg GDP per Capita (US$)",
    legend_title="Income Group",
    height=700,
    width=1100,
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(l=60, r=30, t=120, b=110)
)

fig.show()

Breakout Year: 1985
Std gap at breakout: $4,199


<h3>Question 2: Which countries had the highest GDP per capita growth (1960–2024)?</h3>

<h5>The analytical task is to rank and compare absolute GDP per capita growth in USD across countries
over a 64-year period. A horizontal bar chart is a strong fit because it excels at ranking comparisons
across many categories — the length of each bar directly encodes the magnitude of growth, making it
easy to compare countries at a glance. Bars are sorted from highest to lowest so the ranking is
immediately clear. An alternative would be a line chart. However, it would be weaker here because
with 15 countries plotted over 64 years, the chart becomes cluttered and the final ranking is harder
to read than a simple sorted bar. The key takeaway is that Bermuda, Luxembourg and Ireland dominate
by absolute USD growth, reflecting decades of financial services and economic development, while all
15 countries grew well above the world average of ~$13,000.</h5>


In [96]:
# ── Top 15 Countries by GDP per Capita Growth in USD (1960–2024) ──────────────

df_1960 = df1[df1["Year"] == 1960][["Country Name", "GDP_per_capita"]].rename(columns={"GDP_per_capita": "gdp_1960"})
df_2024 = df1[df1["Year"] == 2024][["Country Name", "GDP_per_capita"]].rename(columns={"GDP_per_capita": "gdp_2024"})

merged = pd.merge(df_1960, df_2024, on="Country Name")
merged["growth_usd"] = merged["gdp_2024"] - merged["gdp_1960"]

top15 = merged.sort_values("growth_usd", ascending=False).head(15)
world_avg_usd = merged["growth_usd"].mean()

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = px.bar(
    top15.sort_values("growth_usd", ascending=True),
    x="growth_usd",
    y="Country Name",
    orientation="h",
    title="Top 15 Countries by GDP per Capita Growth (1960–2024)",
    labels={"growth_usd": "GDP per Capita Increase (US$)", "Country Name": "Country"},
    text="growth_usd",
    
    color_discrete_sequence=["#4B5EFA"]
    
)

fig.update_traces(
    texttemplate="$%{text:,.0f}",
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Growth: $%{x:,.0f}<extra></extra>"
)

fig.add_vline(
    x=world_avg_usd,
    line_dash="dash",
    line_color="black",
    annotation_text="World avg",
    annotation_position="top"
)

# ── Subtitle text below title ─────────────────────────────────────────────────
fig.add_annotation(
    x=0,
    y=1.14,
    xref="paper",
    yref="paper",
    text=(
        "From 1960 to 2024, Bermuda led with $141,140 growth while the world average was ~$13,000.<br>"
        "Western nations dominate by absolute USD growth — Bermuda, Luxembourg and Ireland top the list.<br>"
        "All 15 countries grew well above the world average, reflecting decades of sustained development."
    ),
    showarrow=False,
    font=dict(size=13, color="#2f4f6f"),
    align="left",
    xanchor="left"
)

# ── Bottom takeaway ───────────────────────────────────────────────────────────
fig.add_annotation(
    x=0.5,
    y=-0.2,
    xref="paper",
    yref="paper",
    text=(
        "<b>Takeaway:</b> Bermuda, Luxembourg and Ireland grew the most in absolute USD terms since 1960. "
        "<br>High-income Western nations dominate — reflecting compounding wealth advantages over 64 years."
    ),
    showarrow=False,
    font=dict(size=11, color="gray"),
    align="center",
    xanchor="center"
)

fig.update_layout(
    title=dict(
        text="Top 15 Countries by GDP per Capita Growth (1960–2024)",
        x=0.11,
        xanchor="left",
        y=0.93,
        yanchor="top",
        font=dict(size=20)    # ← change this number
    ),
    height=900,
    width=1000,
    xaxis_title="GDP per Capita Increase (US$)",
    yaxis_title="Country",
    uniformtext_minsize=8,
    uniformtext_mode="hide",
    xaxis_range=[0, merged["growth_usd"].max() * 1.2],
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(l=40, r=20, t=180, b=140)
)

fig.show()

<h3>Question 2:  Which countries had the highest GDP per capita growth (1960–2024)?</h3>

<h5>This chart uses the same title, format and color as the truth version but 
silently displays Australia (AUD), Canada (CAD), New Zealand (NZD), 
Saudi Arabia (SAR) and China (CNY) in local currency instead of USD — 
without any disclosure. Saudi Arabia appears at 129,014 and China at 93,815, 
making them look competitive with Western nations when their real USD growth 
is only $34,404 and $13,213 respectively. The world average line is also 
inflated by 1.5x to match the distorted scale. No underlying data values 
were changed — the deception is purely in the unit of display.</h5>

In [97]:
# ══════════════════════════════════════════════════════════════════════════════
# LIE: Mix USD and local currencies — Australia, Canada, New Zealand,
# Singapore, Saudi Arabia and China shown in local currency without disclosure
# ══════════════════════════════════════════════════════════════════════════════

df_1960 = df1[df1["Year"] == 1960][["Country Name", "GDP_per_capita"]].rename(columns={"GDP_per_capita": "gdp_1960"})
df_2024 = df1[df1["Year"] == 2024][["Country Name", "GDP_per_capita"]].rename(columns={"GDP_per_capita": "gdp_2024"})
merged = pd.merge(df_1960, df_2024, on="Country Name")
merged["growth_usd"] = merged["gdp_2024"] - merged["gdp_1960"]

# ── Real world avg in USD ─────────────────────────────────────────────────────
real_world_avg = merged["growth_usd"].mean()

# ── Top 10 by USD + 5 extra = 15 total ───────────────────────────────────────
top10_usd = merged.sort_values("growth_usd", ascending=False).head(10)
extra     = merged[merged["Country Name"].isin([
    "Australia", "Canada", "New Zealand", "Saudi Arabia", "China"
])]
df_mixed  = pd.concat([top10_usd, extra]).drop_duplicates(subset="Country Name").reset_index(drop=True)

# ── Exchange rates (2024) ─────────────────────────────────────────────────────
exchange_rates = {
    "Australia":    1.53,
    "Canada":       1.36,
    "New Zealand":  1.63,
    "Saudi Arabia": 3.75,
    "China":        7.10,
    "Singapore":    1.34,
}

currency_labels = {
    "Australia":    "AUD",
    "Canada":       "CAD",
    "New Zealand":  "NZD",
    "Saudi Arabia": "SAR",
    "China":        "CNY",
    "Singapore":    "SGD",
}

df_mixed["exchange_rate"] = df_mixed["Country Name"].map(exchange_rates).fillna(1.0)
df_mixed["currency"]      = df_mixed["Country Name"].map(currency_labels).fillna("USD")
df_mixed["growth_mixed"]  = df_mixed["growth_usd"] * df_mixed["exchange_rate"]
df_mixed = df_mixed.sort_values("growth_mixed", ascending=True).reset_index(drop=True)

# ── Fake world avg ────────────────────────────────────────────────────────────
fake_world_avg = real_world_avg * 1.5

# ── Plot ──────────────────────────────────────────────────────────────────────
fig_lie = px.bar(
    df_mixed,
    x="growth_mixed",
    y="Country Name",
    orientation="h",
    labels={"growth_mixed": "GDP per Capita Increase", "Country Name": "Country"},
    text="growth_mixed",
    color_discrete_sequence=["#4B5EFA"]
)

fig_lie.update_traces(
    texttemplate="%{text:,.0f}",
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Growth: %{x:,.0f}<extra></extra>"
)

fig_lie.add_vline(
    x=fake_world_avg,
    line_dash="dash",
    line_color="black",
    annotation_text="World avg",
    annotation_position="top"
)

# ── Subtitle text — fake, sounds legitimate ───────────────────────────────────
fig_lie.add_annotation(
    x=0,
    y=1.14,
    xref="paper",
    yref="paper",
    text=(
        "From 1960 to 2024, GDP per capita growth varied significantly across countries.<br>"
        "Emerging economies like Saudi Arabia and China show remarkable gains alongside Western nations.<br>"
        "Australia, Canada and New Zealand demonstrate strong consistent long-term economic development."
    ),
    showarrow=False,
    font=dict(size=13, color="#2f4f6f"),
    align="left",
    xanchor="left"
)

# ── Bottom takeaway — fake, sounds legitimate ─────────────────────────────────
fig_lie.add_annotation(
    x=0.5,
    y=-0.2,
    xref="paper",
    yref="paper",
    text=(
        "<b>Takeaway:</b> Saudi Arabia and China have closed the gap with traditional Western economies. "
        "<br>Asia-Pacific nations show strong broad-based growth, signaling a major global economic shift since 1960."
    ),
    showarrow=False,
    font=dict(size=11, color="gray"),
    align="center",
    xanchor="center"
)

fig_lie.update_layout(
    title=dict(
        text="Top 15 Countries by GDP per Capita Growth (1960–2024)",
        x=0.11,                 # ← same as truth
        xanchor="left",
        y=0.93,
        yanchor="top",
        font=dict(size=20)
    ),
    height=900,
    width=1000,
    xaxis_title="GDP per Capita Increase",
    yaxis_title="Country",
    uniformtext_minsize=8,
    uniformtext_mode="hide",
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(l=40, r=20, t=180, b=140),
    xaxis_range=[0, df_mixed["growth_mixed"].max() * 1.2]
)

fig_lie.show()